# Experiment 042: GPT-2 124M SFT to GRPO (Colab)

Run of the fixed three seed 124M pipeline. Seed 0 is staged first, and stopped A1 arms are recorded without inventing a GRPO endpoint!

## 0. Compute units and Drive

In [ ]:
NOTEBOOK_NAME = '042_colab_m124_sft_grpo'
import glob, json, os, time

session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 312.65
unit_rate = 5.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': NOTEBOOK_NAME, 'colab_cu_before': cu_before,
                  'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)


session 20260825_164158 | cu_before = 312.65
Mounted at /content/drive
saved open session: 20260825_164158


## 1. Runtime

In [ ]:
import shutil, subprocess, torch

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 39, 'need an A100 40GB or larger runtime'
assert shutil.disk_usage(ROOT).free / 1024**3 >= 35, 'need at least 35 GiB free on Drive'


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-d0e18b84-b2aa-5403-696f-007f94beb3d7)
total vram: 39.5 GiB


## 2. Repository and dependencies

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [ ]:
import gc, os, subprocess, sys, torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
scratch_package = os.path.join(repo_root, '002_GPT2_From_Scratch')
if not os.path.isdir(scratch_package):
    scratch_package = os.path.join(repo_root, '003_My GPT From Scratch')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', scratch_package], check=True)
sys.path.insert(0, scratch_package)
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from gpt2_scratch.weights import REFERENCE_ALIAS, REFERENCE_REVISION
GPT2TokenizerFast.from_pretrained(REFERENCE_ALIAS, revision=REFERENCE_REVISION)
reference = GPT2LMHeadModel.from_pretrained(
    REFERENCE_ALIAS, revision=REFERENCE_REVISION, dtype=torch.float32)
del reference
gc.collect()
print('dependencies installed')


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

dependencies installed


In [ ]:
import getpass, os

wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None
print('W&B key loaded')


W&B key loaded


## 3. Wiring checks

In [ ]:
import os, subprocess, sys

checks = subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    os.path.join(scratch_package, 'tests'),
    'tests/test_scratch_gpt_pipeline.py',
    'tests/test_042_m124.py',
    'tests/test_042_notebooks.py',
    '-k',
    'not test_042_notebooks_are_clean_and_unexecuted and not test_042_units_are_blank_before_launch',
], capture_output=True, text=True)
print(checks.stdout)
if checks.stderr:
    print(checks.stderr)
checks.check_returncode()
print('wiring checks passed')


....................................................................     [100%]
68 passed, 2 deselected in 32.96s

wiring checks passed


In [ ]:
subprocess.run([sys.executable, 'scripts/build_data.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/build_data.py'], returncode=0)

## 4. Registered setup

In [ ]:
import json, os, shutil, subprocess, sys, time
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s42_worker import (BASE_TAG, CONFIGS_042, a1_state, arm_state,
                                arm_tag, commit_a1_conclusion,
                                commit_arm_complete, endpoint_state,
                                pilot_passes, verify_artifact_record)

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# SEEDS_TO_RUN = (0,)
SEEDS_TO_RUN = (0, 1, 2)
# widen to (0, 1, 2) after the first arm completes cleanly
assert set(SEEDS_TO_RUN) <= {0, 1, 2}
cfgs = {seed: load_config(CONFIGS_042[seed]) for seed in (0, 1, 2)}
assert [cfgs[seed].train.seed for seed in (0, 1, 2)] == [0, 1, 2]
assert all(cfg.log.wandb_project == 'inm363-grpo' for cfg in cfgs.values())
os.environ['WANDB_PROJECT'] = cfgs[0].log.wandb_project

TECH = os.path.join(ROOT, 'results', '042_technical')
PILOT_DIR = os.path.join(TECH, 'pilot')
BASE_RESULT = os.path.join(ROOT, 'results', BASE_TAG)
BASE_DIR = os.path.join(ROOT, 'checkpoints', BASE_TAG)
BASE_RECORD_PATH = os.path.join(BASE_RESULT, '042_base_manifest.json')
LENGTH_RECORD = os.path.join(TECH, '042_token_audit.json')
base_record = verify_artifact_record(BASE_RECORD_PATH)
assert read_json(LENGTH_RECORD)['completion_cap'] == 512
pilot_report = read_json(os.path.join(PILOT_DIR, 'pilot_report.json'))
pilot_closeout = read_json(os.path.join(PILOT_DIR, 'pilot_closeout.json'))
assert pilot_passes(pilot_report, pilot_closeout), 'run the technical pilot first'

def append_jsonl(path, row):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a') as f:
        f.write(json.dumps(row) + '\n')

def active_workers(cfg_path):
    process_list = subprocess.run(
        ['ps', '-eo', 'pid=,args='], capture_output=True, text=True, check=True).stdout
    return [line.strip() for line in process_list.splitlines()
            if 'scripts/s42_worker.py' in line and f'--config {cfg_path}' in line]

def run_worker(command, status_path, result_dir, label, test_passes=0):
    if os.path.exists(status_path):
        os.remove(status_path)
    started = time.time()
    process = stream_subprocess(command)
    status = read_json(status_path) if os.path.exists(status_path) else {
        'completed': False, 'error_type': 'MissingStatus',
        'error_message': 'worker exited without a status file'}
    status = {**status, 'label': label, 'returncode': process.returncode,
              'parent_wall_seconds': round(time.time() - started, 1)}
    append_jsonl(os.path.join(result_dir, '042_attempts.jsonl'), status)
    append_jsonl(os.path.join(RESULTS, 'compute_ledger.jsonl'), {
        'run_tag': label, 'condition': f'042_{status.get("task", "worker")}',
        'timings_sec': {'worker': status['parent_wall_seconds']},
        'train_sec': status['parent_wall_seconds'] if status.get('task') in ('sft', 'grpo') else 0,
        'train_steps': status.get('completed_updates', 0),
        'mmlu_test_passes': test_passes,
        'completed': bool(status.get('completed')), 'gpu': status.get('gpu'),
    })
    return process, status

def fresh_scratch(path):
    if os.path.isdir(path) and os.listdir(path):
        os.replace(path, path + '_failed_' + time.strftime('%Y%m%d_%H%M%S'))
    os.makedirs(path, exist_ok=True)
    return path

print('seeds selected:', SEEDS_TO_RUN)


seeds selected: (0, 1, 2)


## 5. Shared base evaluation

In [ ]:
def run_endpoint(cfg_path, branch, model_path, record_path, result_dir, label):
    binding = read_json(record_path)
    state = endpoint_state(result_dir, branch, binding)
    if state == 'tampered':
        raise RuntimeError(f'committed {branch} endpoint changed')
    if state == 'committed':
        return read_json(os.path.join(result_dir, f'042_{branch}_eval_manifest.json'))
    scratch = fresh_scratch(f'/content/042_eval_{label}')
    status_path = os.path.join(result_dir, f'{branch}_eval_status.json')
    command = [
        sys.executable, 'scripts/s42_worker.py', '--task', 'eval',
        '--config', cfg_path, '--status', status_path,
        '--result-dir', result_dir, '--start-dir', model_path,
        '--scratch-dir', scratch, '--record', record_path, '--branch', branch,
    ]
    process, status = run_worker(
        command, status_path, result_dir, label, test_passes=2)
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'{branch} evaluation failed: {status}')
    shutil.rmtree(scratch)
    return read_json(os.path.join(result_dir, f'042_{branch}_eval_manifest.json'))

base_endpoint = run_endpoint(
    CONFIGS_042[0], 'base', BASE_DIR, BASE_RECORD_PATH,
    BASE_RESULT, '042_m124_base_eval')
print('base endpoint committed:', base_endpoint['completed'])


base endpoint committed: True


## 6. Seeded pipelines

In [ ]:
def arm_paths(seed):
    tag = arm_tag(seed)
    result_dir = os.path.join(ROOT, 'results', tag)
    checkpoint_root = os.path.join(ROOT, 'checkpoints', tag)
    return result_dir, os.path.join(checkpoint_root, 'sft'), os.path.join(checkpoint_root, 'grpo')

def run_training(seed, task, result_dir, output_dir, start_dir, status_name):
    cfg_path = CONFIGS_042[seed]
    record_name = '042_sft_training.json' if task == 'sft' else '042_grpo_training.json'
    record_path = os.path.join(result_dir, record_name)
    if os.path.exists(record_path):
        return read_json(record_path)
    status_path = os.path.join(result_dir, status_name)
    if os.path.exists(status_path):
        previous = read_json(status_path)
        if previous.get('error_type') != 'KeyboardInterrupt':
            return previous
    active = active_workers(cfg_path)
    assert not active, f'experiment 042 worker already active: {active}'
    command = [
        sys.executable, 'scripts/s42_worker.py', '--task', task,
        '--config', cfg_path, '--status', status_path,
        '--result-dir', result_dir, '--out-dir', output_dir,
    ]
    if task == 'sft':
        command.extend(['--base-dir', start_dir])
    else:
        command.extend(['--start-dir', start_dir, '--precision-report',
                        os.path.join(result_dir, 'precision_report.json')])
    process, status = run_worker(
        command, status_path, result_dir, f'042_m124_seed{seed}_{task}')
    if process.returncode == 0 and status.get('completed'):
        return read_json(record_path)
    return status

def run_arm(seed):
    result_dir, sft_dir, grpo_dir = arm_paths(seed)
    os.makedirs(result_dir, exist_ok=True)
    current = arm_state(result_dir)
    if current['state'] == 'committed':
        return 'committed'
    if current['state'] == 'failed':
        raise RuntimeError(f'seed {seed} has a previous GRPO failure: {current["status"]}')

    sft_record = run_training(
        seed, 'sft', result_dir, sft_dir, BASE_DIR, 'sft_worker_status.json')
    if sft_record.get('completed') is not True or sft_record.get('stage') != 'sft':
        raise RuntimeError(f'SFT failed for seed {seed}: {sft_record}')

    a1, _ = a1_state(result_dir)
    if a1 == 'pending':
        commit_a1_conclusion(result_dir)
        a1 = 'committed'
    if a1 == 'failed':
        raise RuntimeError(f'seed {seed} has a GRPO failure other than A1')
    if a1 not in ('committed',):
        grpo_record = run_training(
            seed, 'grpo', result_dir, grpo_dir, sft_dir, 'grpo_worker_status.json')
        a1, _ = a1_state(result_dir)
        if a1 == 'pending':
            commit_a1_conclusion(result_dir)
            a1 = 'committed'
        elif grpo_record.get('completed') is not True or grpo_record.get('stage') != 'grpo':
            raise RuntimeError(f'GRPO failed for seed {seed}: {grpo_record}')

    run_endpoint(
        CONFIGS_042[seed], 'sft', sft_dir,
        os.path.join(result_dir, '042_sft_training.json'), result_dir,
        f'042_m124_seed{seed}_sft_eval')
    if a1 == 'committed':
        return 'a1'
    run_endpoint(
        CONFIGS_042[seed], 'grpo', grpo_dir,
        os.path.join(result_dir, '042_grpo_training.json'), result_dir,
        f'042_m124_seed{seed}_grpo_eval')
    commit_arm_complete(result_dir)
    return 'committed'

outcomes = {seed: run_arm(seed) for seed in SEEDS_TO_RUN}
print('session outcomes:', outcomes)


Streaming output truncated to the last 5000 lines.
│ │ where $a$ is  │               │               │              │           │ │
│ │ an integer,   │               │               │              │           │ │
│ │ and $b$ is a  │               │               │              │           │ │
│ │ positive      │               │               │              │           │ │
│ │ integer. If   │               │               │              │           │ │
│ │ $b$ is as     │               │               │              │           │ │
│ │ small as      │               │               │              │           │ │
│ │ possible,     │               │               │              │           │ │
│ │ then what is  │               │               │              │           │ │
│ │ $a+b$?        │               │               │              │           │ │
│ │ A. 2          │               │               │              │           │ │
│ │ B. -4         │               │               │       

## 7. Current state

In [ ]:
states = {seed: arm_state(arm_paths(seed)[0])['state'] for seed in (0, 1, 2)}
print('all registered seeds:', states)
completed_pairs = sum(state == 'committed' for state in states.values())
print('completed SFT to GRPO pairs:', completed_pairs)
if set(SEEDS_TO_RUN) == {0} and states[0] in ('committed', 'a1'):
    print('seed 0 settled; widen SEEDS_TO_RUN to (0, 1, 2) for the remaining arms')


all registered seeds: {0: 'committed', 1: 'committed', 2: 'committed'}
completed SFT to GRPO pairs: 3


## 8. Close compute units

In [14]:
cu_after = 288.62
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open session record')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)


cu_after = 288.62
logged: {'session_id': '20260825_164158', 'ts_start': '2026-08-25 16:42', 'notebook': '042_colab_m124_sft_grpo', 'colab_cu_before': 312.65, 'unit_rate': 5.3, 'ts_end': '2026-08-25 21:21', 'colab_cu_after': 288.62, 'colab_cu_used': 24.03, 'completed': True}
